# ⏰ Japan Grid Analysis: Temporal Surplus Patterns

**Notebook 03:** Identify when surplus windows occur across hours, days, and seasons.

This notebook answers: **When** do surplus windows occur?

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Load cleaned data
df = pd.read_csv('../data/processed/japan_grid_clean.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df['hour'] = df['datetime'].dt.hour
df['month'] = df['datetime'].dt.month
df['day_of_week'] = df['datetime'].dt.weekday  # 0=Mon, 6=Sun
df['season'] = df['month'].apply(lambda x: 
    'Winter' if x in [12, 1, 2] else
    'Spring' if x in [3, 4, 5] else
    'Summer' if x in [6, 7, 8] else
    'Autumn')

print(f"✅ Data prepared: {len(df):,} records")
print(f"Seasons: {df['season'].unique()}")

## Chart 1: Hourly Surplus Profile by Season

In [ ]:
# Average surplus by hour and season (across all regions)
hourly_seasonal = df.groupby(['hour', 'season'])['surplus_mw'].mean().reset_index()

# Order seasons properly
season_order = ['Winter', 'Spring', 'Summer', 'Autumn']
season_colors = {'Winter': 'lightblue', 'Spring': 'lightgreen', 'Summer': 'gold', 'Autumn': 'orange'}

fig = go.Figure()

for season in season_order:
    season_data = hourly_seasonal[hourly_seasonal['season'] == season]
    fig.add_trace(go.Scatter(
        x=season_data['hour'],
        y=season_data['surplus_mw'],
        name=season,
        mode='lines+markers',
        line=dict(width=2),
        marker=dict(size=6)
    ))

# Shade peak surplus window
fig.add_vrect(x0=10, x1=14, fillcolor='green', opacity=0.1, layer='below', annotation_text='Peak Surplus Window')

fig.update_layout(
    title="Average Hourly Renewable Surplus by Season (All Regions)",
    xaxis_title="Hour of Day (JST)",
    yaxis_title="Average Surplus (MW)",
    xaxis=dict(tickmode='linear', tick0=0, dtick=1),
    height=500,
    template='plotly_white',
    hovermode='x unified'
)

fig.show()
print("✅ Hourly seasonal profile generated")

## Chart 2: Weekly Heatmap (Day × Hour Patterns)

In [ ]:
# Average surplus by day-of-week and hour
weekly_hourly = df.groupby(['day_of_week', 'hour'])['surplus_mw'].mean().reset_index()
heatmap_data = weekly_hourly.pivot(index='day_of_week', columns='hour', values='surplus_mw')

day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns,
    y=[day_names[i] for i in heatmap_data.index],
    colorscale='RdYlGn',
    zmid=0,
    colorbar=dict(title="Surplus<br>(MW)")
))

fig.update_layout(
    title="Surplus Patterns by Day of Week & Hour",
    xaxis_title="Hour of Day (JST)",
    yaxis_title="Day of Week",
    height=400,
    template='plotly_white'
)

fig.show()
print("✅ Weekly heatmap generated")

## Chart 3: Monthly Surplus Distribution (Box Plot)

In [ ]:
# Create month names for plotting
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_to_season = {
    1: 'Winter', 2: 'Winter', 3: 'Spring',
    4: 'Spring', 5: 'Spring', 6: 'Summer',
    7: 'Summer', 8: 'Summer', 9: 'Autumn',
    10: 'Autumn', 11: 'Autumn', 12: 'Winter'
}
df['season_from_month'] = df['month'].map(month_to_season)

fig = go.Figure()

for month in range(1, 13):
    month_data = df[df['month'] == month]['surplus_mw']
    season = month_to_season[month]
    season_colors_map = {'Winter': 'lightblue', 'Spring': 'lightgreen', 'Summer': 'gold', 'Autumn': 'orange'}
    
    fig.add_trace(go.Box(
        y=month_data,
        name=month_names[month-1],
        marker=dict(color=season_colors_map[season]),
        boxmean='sd'
    ))

fig.update_layout(
    title="Monthly Distribution of Renewable Surplus",
    xaxis_title="Month",
    yaxis_title="Surplus (MW)",
    height=500,
    template='plotly_white',
    showlegend=False
)

fig.show()
print("✅ Monthly box plot generated")

## 🔍 Key Temporal Insights

In [ ]:
# Find peak surplus window (month-hour combination)
monthly_hourly = df.groupby(['month', 'hour'])['surplus_mw'].mean().reset_index()
peak_row = monthly_hourly.loc[monthly_hourly['surplus_mw'].idxmax()]

month_names_map = {1: 'January', 2: 'February', 3: 'March', 4: 'April', 5: 'May', 6: 'June',
                   7: 'July', 8: 'August', 9: 'September', 10: 'October', 11: 'November', 12: 'December'}

print("\n⏰ PEAK SURPLUS WINDOW:")
print(f"Month: {month_names_map[int(peak_row['month'])]}")
print(f"Hour: {int(peak_row['hour'])}:00–{int(peak_row['hour'])+1}:00 JST")
print(f"Average surplus: {peak_row['surplus_mw']:.1f} MW")

# Worst curtailment months
monthly_avg = df.groupby('month')['surplus_mw'].mean().sort_values(ascending=False)
print("\n📊 TOP 3 SURPLUS MONTHS:")
for i, (month, surplus) in enumerate(monthly_avg.head(3).items(), 1):
    print(f"{i}. {month_names_map[int(month)]}: {surplus:.1f} MW average") 

print("\n🛢️ LEAST SURPLUS MONTHS:")
for i, (month, surplus) in enumerate(monthly_avg.tail(3).items(), 1):
    print(f"{i}. {month_names_map[int(month)]}: {surplus:.1f} MW average")

# Hourly analysis
hourly_avg = df.groupby('hour')['surplus_mw'].mean().sort_values(ascending=False)
print("\n🎯 PEAK SURPLUS HOURS (top 5):")
for i, (hour, surplus) in enumerate(hourly_avg.head(5).items(), 1):
    print(f"{i}. {int(hour):02d}:00 JST: {surplus:.1f} MW average")